In [1]:
!pip install -q langchain_community faiss-gpu langchain-huggingface langchain-anthropic pytube youtube-transcript-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.3/622.3 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.5/199.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.5/409.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.0 MB/s eta 0:00:00


In [2]:
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

from langchain_core.prompts import ChatPromptTemplate

from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import YoutubeLoader
from langchain_community.document_loaders.youtube import TranscriptFormat

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic

In [3]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

In [4]:
model = "claude-3-5-sonnet-20241022"
temperature = 0
max_tokens = 1000

In [5]:
url_video = "https://www.youtube.com/watch?v=vH2f7cjXjKI"
video_loader = YoutubeLoader.from_youtube_url(
              url_video,
              language=["pt", "pt-BR", "en"],
              translation="pt",

              )

docs = video_loader.load()

transcript = docs[0].page_content

In [6]:
transcript

'Meu nome é Alex, lidero\nrelações com desenvolvedores na Anthropic e hoje vou\nmostrar a vocês uma tarefa de codificação com uso de computador. Então, mostraremos Claude\nrealizando uma tarefa de codificação de site controlando meu laptop. Mas antes de começarmos a programar, precisamos de um site real para\nClaude fazer alterações. Então, vamos pedir ao Claude para\nnavegar até claude.ai no meu navegador Chrome e pedir ao Claude do Claude.ai para criar uma página pessoal divertida com o tema dos anos 90\npara si. Claude abre o Chrome, pesquisa Claude.ai e digita um prompt\npedindo ao outro Claude para criar uma página inicial pessoal para si. Claude.ai retorna algum código, e ele é bem renderizado em um Artefato no lado direito. Parece ótimo, mas quero fazer algumas\nalterações no site localmente no meu próprio computador. Vamos pedir ao Claude para baixar o arquivo e depois abri-lo no VS Code. Claude clica no botão salvar em arquivo, abre o VS Code e, em seguida, encontra o arquivo\

In [10]:
def load_video(url_video, language=["pt", "pt-BR", "en"], translation="pt"):
  video_loader = YoutubeLoader.from_youtube_url(
      url_video,
      language=language,
      translation=translation,
      )

  docs = video_loader.load()

  # transcript = docs[0].page_content

  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

  vector_store = FAISS.from_documents(docs, embeddings)

  return vector_store

transcript = load_video(url_video)

In [8]:
transcript

'Meu nome é Alex, lidero\nrelações com desenvolvedores na Anthropic e hoje vou\nmostrar a vocês uma tarefa de codificação com uso de computador. Então, mostraremos Claude\nrealizando uma tarefa de codificação de site controlando meu laptop. Mas antes de começarmos a programar, precisamos de um site real para\nClaude fazer alterações. Então, vamos pedir ao Claude para\nnavegar até claude.ai no meu navegador Chrome e pedir ao Claude do Claude.ai para criar uma página pessoal divertida com o tema dos anos 90\npara si. Claude abre o Chrome, pesquisa Claude.ai e digita um prompt\npedindo ao outro Claude para criar uma página inicial pessoal para si. Claude.ai retorna algum código, e ele é bem renderizado em um Artefato no lado direito. Parece ótimo, mas quero fazer algumas\nalterações no site localmente no meu próprio computador. Vamos pedir ao Claude para baixar o arquivo e depois abri-lo no VS Code. Claude clica no botão salvar em arquivo, abre o VS Code e, em seguida, encontra o arquivo\

In [ ]:
def load_video(url_video, language=["pt", "pt-BR", "en"], translation="pt"):
  video_loader = YoutubeLoader.from_youtube_url(
      url_video,
      language=language,
      translation=translation,
      )

  docs = video_loader.load()

  transcript = docs[0].page_content

  return transcript

def get_response(query, transcript, llm):

  template="""
    Você é um assistente virtual encarregado de executar uma tarefa
    solicitada pelo usuário com base na transcricao que foi extraído
    de um video do YouTube, que serao fornecidos abaixo.
    Seu objetivo é analisar essa transcricao e concluir
    a tarefa especificada pelo usuário.

    Aqui está a trancricao do vídeo:
    <transcricao>
    {transcript}
    </transcricao>

    Aqui está a tarefa que o usuário deseja que você execute:
    <query>
    {query}
    </query>

    Se você não receber uma transcricao, alerte o usuário respondendo com:
    "Sinto muito, mas não recebi uma transcricao para trabalhar.
    Você poderia fornecer a transcricao do vídeo?"

    Se você recebeu um contexto e a consulta, siga estas etapas:

    1. Leia e analise cuidadosamente a transcricao.
    2. Entenda a consulta do usuário e a tarefa que ele deseja que você execute.
    3. Execute a tarefa com base nas informações disponíveis na transcricao.
    4. Se a transcricao não contiver informações suficientes para concluir a tarefa,
    explique quais informações estão faltando e por que você não consegue executar
    a tarefa completamente.

  """

  prompt_template = PromptTemplate(
    input_variables=["context", "query"],
    template=template)

  memory = ConversationBufferMemory(memory_key="chat_history",
                                    input_key="query")

  # Criar uma chain
  chain = LLMChain(
      llm=llm,
      prompt=prompt_template,
      memory=memory
  )

  response = chain.invoke({"transcript": transcript,
                           "query": query})
  return response["text"]

In [ ]:
llm = ChatAnthropic(model=model,
                      temperature=temperature,
                      max_tokens=max_tokens)

In [ ]:
url_video = "https://www.youtube.com/watch?v=RoR4XJw8wIc"
transcript = load_video(url_video)

In [ ]:
query = "Quais os pontos mais importantes do video?"
response = get_response(query, transcript, llm)
print(response)

Com base na transcrição fornecida, aqui estão os pontos mais importantes do vídeo:

1. Sobre o LangChain:
- É um dos frameworks Python mais recentes, lançado no final do ano passado
- Já possui mais de 20.000 estrelas no GitHub
- Recentemente recebeu um investimento de 10 milhões de dólares
- Possui uma comunidade ativa de contribuidores

2. Propósito:
- É uma estrutura para desenvolvimento de aplicativos alimentados por grandes modelos de linguagem
- Permite integrar modelos de linguagem com diferentes fontes de dados e funcionalidades

3. Principais funcionalidades (6 módulos):
- Interface genérica para diversos LLMs (modelos de linguagem)
- Gerenciamento de prompts
- Cadeias (chains) para sequenciar operações
- Sistema de memória para armazenar históricos de conversas
- Índices para combinar modelos com dados próprios (PDFs, e-mails, etc.)
- Agentes e ferramentas (permite integração com Google, Wikipedia, calculadoras, etc.)

4. Benefícios:
- Simplifica a construção de aplicativos c